### Warm-up: How to grab sea surface temperature near Houston

The old way to work with gridded data is to keep one array of values, one array of latitudes, and one array of longitudes. That works, but it puts the bookkeeping on you.

Run the cell below and notice the two-step process: first find the row and column, then use those positions to get data.


In [ ]:
import numpy as np

# Pretend we loaded a 720x1440 SST grid.
# Lat runs from -89.875 to 89.875 in 0.25° steps.
# Lon runs from 0.125 to 359.875 in 0.25° steps.
lats = np.arange(-89.875, 90, 0.25)
lons = np.arange(0.125, 360, 0.25)

# I want lat ≈ 29°N, lon ≈ 265°E (Gulf of Mexico, near Houston).
lat_idx = np.argmin(np.abs(lats - 29.0))
lon_idx = np.argmin(np.abs(lons - 265.0))

print(f'Raw array: lat index={lat_idx}, lon index={lon_idx}')
print(f'That corresponds to lat={lats[lat_idx]:.3f}, lon={lons[lon_idx]:.3f}')


# Lesson 4: Gridded Data with xarray

Today's mental model:

- A **Dataset** is like a library. It holds several related data variables and their shared coordinates.
- A **DataArray** is like a checked-out book. It is one variable you can read, plot, subset, and compute with directly.

We will open one NetCDF file as a Dataset, immediately check out sea surface temperature as a DataArray, and spend most of the lesson working with that DataArray.

### Core goals

By the end of this session you should be able to:

- Explain the difference between a **Dataset** and a **DataArray** using the library/book analogy.
- Identify the three useful parts of a DataArray: **values**, **dims**, and **coords**.
- Open a NetCDF file with `xr.open_dataset()` and check out one variable with `ds['variable_name']`.
- Use `.isel()`, `.sel()`, `.mean(dim=...)`, `.where()`, and `.plot()` on one DataArray.


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt


## Block 1: Why labeled arrays help

A DataArray keeps the numbers and their labels together.

```
DataArray
├── values  - the actual numbers, like a numpy array
├── dims    - names of the axes, like ('lat', 'lon')
└── coords  - coordinate values along those axes, like latitude and longitude
```

This is the main point for today: once coordinates are attached, you can ask for `lat=29.0` instead of hunting for the integer row number yourself.


In [ ]:
# Build a small DataArray from scratch to see the anatomy.
temps = np.array([[15.1, 16.2, 17.3, 18.0],
                  [20.5, 21.8, 22.1, 23.4],
                  [26.0, 27.2, 28.5, 29.1]])

da = xr.DataArray(
    data=temps,
    dims=['lat', 'lon'],
    coords={'lat': [20, 25, 30],
            'lon': [260, 265, 270, 275]},
    name='sst',
    attrs={'units': 'Celsius', 'long_name': 'Sea Surface Temperature'}
)

da


### Discussion: raw array plot vs DataArray plot

First plot the raw numpy array. Then plot the DataArray. With a neighbor, compare what information appears in each plot.


In [ ]:
plt.imshow(temps)
plt.title('Raw numpy array')
plt.colorbar(label='temperature')
plt.show()


In [ ]:
da.plot()
plt.title('Labeled xarray DataArray')
plt.show()


In [ ]:
print('dims:  ', da.dims)
print('coords:', list(da.coords))
print('attrs: ', da.attrs)
print()
print('The raw numbers underneath:')
print(da.values)


In [ ]:
# Select a single latitude by coordinate label.
da.sel(lat=25).plot(marker='o')
plt.title('Temperature along latitude 25°')
plt.show()


### Activity 1: Build your own DataArray

Take about 5 minutes. The goal is not to memorize the syntax. The goal is to see where `data`, `dims`, `coords`, and `attrs` go.


In [ ]:
# Activity 1 - Build a labeled DataArray
# Goal: make a 3-day temperature time series for one location.
# Use these pieces:
#   data = [18.5, 19.2, 17.8]
#   dims = ['time']
#   coords = {'time': ['2022-01-01', '2022-01-02', '2022-01-03']}
# Add units in attrs, then plot it with marker='o'.

temps_ts = xr.DataArray(
    data=[18.5, 19.2, 17.8],
    dims=['time'],
    coords={'time': ['2022-01-01', '2022-01-02', '2022-01-03']},
    name='temperature',
    attrs={'units': 'Celsius'}
)

print(temps_ts)
temps_ts.plot(marker='o')
plt.title('3-day temperature time series')
plt.show()


## Block 2: Open a real NetCDF file, then check out one DataArray

In real work, you usually do not build DataArrays from scratch. You open a file.

The NetCDF file opens as a **Dataset** because it contains multiple variables. For this lesson, we will treat the Dataset as the library, check out `sst`, and then do our analysis on the checked-out DataArray.


In [ ]:
ds = xr.open_dataset('data/oisst_sst_20220304.nc')
ds


In [ ]:
print('Dataset dimensions:', dict(ds.sizes))
print('Variables in the library:', list(ds.data_vars))
print('Coordinates:', list(ds.coords))


### Check out SST

From here forward, most of our work uses `sst_2d`, not the full Dataset.


In [ ]:
sst = ds['sst']
print(sst)


The file has one time and one depth level, so the raw SST DataArray has extra size-1 dimensions. `.squeeze()` removes those size-1 dimensions and gives us a clean 2D latitude-by-longitude DataArray.


In [ ]:
sst_2d = sst.squeeze()
print('Before squeeze:', sst.dims, sst.shape)
print('After squeeze: ', sst_2d.dims, sst_2d.shape)


In [ ]:
h = sst_2d.plot(cmap='RdYlBu_r', figsize=(12, 5))
plt.title('Global Sea Surface Temperature - March 4, 2022')
h.set_clim(-2, 35)
plt.show()


### Activity 2: Check your checked-out DataArray

Use `sst_2d` for the coordinate questions. If you look at another variable, check it out as its own DataArray first.


In [ ]:
# Activity 2 - Coordinate detective
# Do not stop after getting numbers. Use the numbers to answer the questions in comments.
#
# 1. Prove this is a global grid:
#    - What are the lat/lon min and max?
#    - Does longitude use -180 to 180 or 0 to 360?
#
# 2. Prove this is a 0.25° grid:
#    - What is the difference between the first two lat values?
#    - What is the difference between the first two lon values?
#
# 3. Explain why the latitude range is -89.875 to 89.875 instead of -90 to 90.
#    Hint: are these grid-cell centers or grid-cell edges?
#
# 4. Check out the ice variable as a DataArray and plot it.
#    In one sentence, describe where the nonzero sea ice appears.

print('Lat range:', float(sst_2d.lat.min()), 'to', float(sst_2d.lat.max()), '°N')
print('Lon range:', float(sst_2d.lon.min()), 'to', float(sst_2d.lon.max()), '°E')

lat_step = float(sst_2d.lat[1] - sst_2d.lat[0])
lon_step = float(sst_2d.lon[1] - sst_2d.lon[0])
print(f'Lat step: {lat_step:.3f}°')
print(f'Lon step: {lon_step:.3f}°')

# Interpretation:
# This is a global 0.25° grid. Longitude uses 0 to 360, not -180 to 180.
# The grid stops at +/-89.875 because these coordinates are grid-cell centers.
# A 0.25° cell centered at 89.875 extends to 90.0 at its northern edge.

ice = ds['ice'].squeeze()
ice.plot(figsize=(12, 5), cmap='Blues')
plt.title('Sea Ice Concentration - March 4, 2022')
plt.show()

# Nonzero sea ice appears mainly near the poles.


## Block 3: Select, compute, mask, plot

Now that we have one checked-out DataArray, the main tools are:

| Method | Meaning | Example |
|--------|---------|---------|
| `.isel()` | Select by integer position | `sst_2d.isel(lat=0)` |
| `.sel()` | Select by coordinate value | `sst_2d.sel(lat=0.0, method='nearest')` |
| `.mean(dim=...)` | Average over named dimensions | `sst_2d.mean(dim='lon')` |
| `.where()` | Keep values where a condition is true | `sst_2d.where(sst_2d > 20)` |
| `.plot()` | Quick plot using attached coordinates | `sst_2d.plot()` |


In [ ]:
# .isel(): select by integer position.
first_row = sst_2d.isel(lat=0)
print('lat value of row 0:', float(first_row.lat), '°N')

first_row.plot(marker='o', markersize=2, linewidth=0.8, figsize=(12, 3))
plt.title(f'SST at lat = {float(first_row.lat):.3f}°N (row 0)')
plt.show()


In [ ]:
# .sel(): select by coordinate value.
# method='nearest' finds the closest grid point if the exact value is not in the grid.
equator = sst_2d.sel(lat=0.0, method='nearest')
print('lat value matched:', float(equator.lat), '°N')

equator.plot(marker='o', markersize=2, linewidth=0.8, figsize=(12, 3))
plt.title('Equatorial SST - March 4, 2022')
plt.show()


In [ ]:
# .sel() with slice() selects a coordinate range.
# Gulf of Mexico region: lat 18-31°N, lon 262-278°E.
gulf = sst_2d.sel(lat=slice(18, 31), lon=slice(262, 278))

print('Gulf of Mexico subset:')
print('  Shape:', gulf.shape)
print('  Lat range:', float(gulf.lat.min()), 'to', float(gulf.lat.max()))
print('  Lon range:', float(gulf.lon.min()), 'to', float(gulf.lon.max()))

gulf.plot(cmap='RdYlBu_r', figsize=(8, 5))
plt.title('Gulf of Mexico SST - March 4, 2022')
plt.show()


### Activity 3: Debug a coordinate selection

This is a common xarray error: asking for a coordinate value that is close to the grid, but not exactly on the grid.


In [ ]:
# Activity 3 - Debug a coordinate selection
# This line fails because 29.0 and 265.0 are not exact coordinate values in this grid.
# Run it if you want to see the error, then fix it below.

# broken = sst_2d.sel(lat=29.0, lon=265.0)

# Fix: tell xarray to use the nearest coordinate values.
fixed = sst_2d.sel(lat=29.0, lon=265.0, method='nearest')
print(f'SST at nearest point: {float(fixed):.2f} °C')
print(f'Matched lat={float(fixed.lat):.3f}, lon={float(fixed.lon):.3f}')


### Computations over named dimensions

Because dimensions have names, you can average over `lon` or `lat` directly. You do not have to remember which axis number means latitude.


In [ ]:
zonal_mean = sst_2d.mean(dim='lon')
print('Input shape: ', sst_2d.shape)
print('Output shape:', zonal_mean.shape)

zonal_mean.plot(figsize=(8, 4))
plt.title('Zonal mean SST, averaged over all longitudes')
plt.ylabel('SST (°C)')
plt.show()


In [ ]:
global_mean = sst_2d.mean(dim=['lat', 'lon'])
print(f'Global mean SST: {float(global_mean):.2f} °C')
print(f'Number of NaN values over land: {int(sst_2d.isnull().sum())}')


In [ ]:
warm_ocean = sst_2d.where(sst_2d > 20)

h = warm_ocean.plot(cmap='hot_r', figsize=(12, 5))
h.set_clim(20, 35)
plt.title('Ocean areas above 20°C - March 4, 2022')
plt.show()


### Activity 4: Regional analysis challenge

Take about 10-12 minutes. This combines the main skills from today: select a region, compute a mean, mask values, and plot the result.


In [ ]:
# Activity 4 - Regional analysis challenge
# Work with the checked-out DataArray sst_2d.
# 1. Select SST for a region of your choice and plot it.
#    Reminder: longitude is 0-360 in this file, not -180 to 180.
# 2. Compute the regional mean SST.
# 3. Use .where() to show only grid cells warmer than the regional mean. Plot it.
# 4. Select a nearby Gulf point close to Houston: 29.0°N, 265.0°E.

# Gulf of Mexico example
region = sst_2d.sel(lat=slice(18, 31), lon=slice(262, 278))

region.plot(cmap='RdYlBu_r', figsize=(8, 5))
plt.title('Gulf of Mexico SST - March 4, 2022')
plt.show()

region_mean = float(region.mean(dim=['lat', 'lon']))
print(f'Regional mean SST: {region_mean:.2f} °C')

region.where(region > region_mean).plot(cmap='RdYlBu_r', figsize=(8, 5))
plt.title(f'Gulf of Mexico SST above regional mean ({region_mean:.1f} °C)')
plt.show()

houston = sst_2d.sel(lat=29.0, lon=265.0, method='nearest')
print(f'SST near the Houston/Gulf point: {float(houston):.2f} °C')
print(f'Matched lat={float(houston.lat):.3f}, lon={float(houston.lon):.3f}')


## 6. Real NASA data: MODIS snow cover (HDF)

So far we've worked with NetCDF. But MODIS products - and many older NASA datasets -
come in HDF4 files. xarray opens them with the same `open_dataset()` call;
you just specify `engine='netcdf4'`.

We have a tile from **MYD10A1F** - MODIS/Aqua **Cloud-Gap-Filled Snow Cover**.
Cloud-gap-filling means the algorithm looks back at recent clear days
to estimate snow cover wherever clouds would otherwise block the view.
The result is a complete spatial picture every day, cloud or not.

This tile is **h11v04** which is mainly Iowa/Illinois/Minnesota/Wisconsin but also covers bits of Ohio/Indiana/Ontario for **February 15, 2026**.

> **SARP connection:** Snow cover drives surface albedo, which affects the surface energy budget
> and boundary layer development. Any project looking at surface–atmosphere exchange
> (aerosol loading, trace gas fluxes, land–air coupling) may need to account for whether
> the surface is snow-covered.

> **Discussion:** Before you look at the data - where would you expect to see the most snow
> in this tile in mid-February? Any areas you'd expect to be patchy or snow-free?

In [ ]:
# Open MODIS HDF4 - same API as NetCDF, just engine='netcdf4'
ds_snow = xr.open_dataset(
    'data/MYD10A1F.A2026046.h11v04.061.2026048035003.hdf',
    engine='netcdf4'
)

# Pull out the cloud-gap-filled snow cover variable
snow = ds_snow['CGF_NDSI_Snow_Cover']
print(snow)
print()
print('Value key:')
print(snow.attrs['Key'])

Notice the dimension names: `YDim:MOD_Grid_Snow_500m` and `XDim:MOD_Grid_Snow_500m`.
That's xarray preserving the HDF metadata, but it's awkward to type.

Also notice: there are **no coordinate values** for those dimensions. MODIS tiles are in
**sinusoidal projection**, not geographic lat/lon. The pixel positions describe a projected grid,
not degrees. For a quick look at the data pattern this is fine; reprojecting to lat/lon is a
separate step.

The other thing to handle: values **above 100 are fill codes**, not snow percentages.
For example, 200 = missing data, 239 = ocean, 250 = cloud. We need to mask those
before computing anything meaningful.

In [ ]:
# Rename the verbose MODIS dimension names to something short
snow = snow.rename({
    'YDim:MOD_Grid_Snow_500m': 'y',
    'XDim:MOD_Grid_Snow_500m': 'x'
})

In [ ]:
# Values 0–100 = NDSI snow cover percent (0 = bare ground, 100 = full snow)
# Values > 100 = fill codes (cloud, water, night, etc.) - mask them
snow_clean = snow.where(snow <= 100)

snow_clean.plot(cmap='Blues_r', figsize=(8, 7))
plt.title('MODIS Aqua Cloud-Gap-Filled Snow Cover\nTile h11v04 (Upper Midwest US) - Feb 15, 2026')
plt.xlabel('x pixel')
plt.ylabel('y pixel')
plt.show()

# Quick summary
n_valid = int(snow_clean.count())
n_total = snow.size
print(f'Valid (snow or bare ground) pixels: {n_valid:,} of {n_total:,} ({100*n_valid/n_total:.0f}%)')

## 7. Bridging xarray and geopandas: adding coordinates to MODIS

Right now our MODIS DataArray has pixel indices for dimensions, which have no geographic meaning.
To overlay lakes, states, or other map data, the snow grid and the vector data need to be
in the same coordinate reference system.

This tile uses the **MODIS sinusoidal projection**. The metadata tells us the upper-left
corner and pixel size in projected meters. We can turn those projected
x/y positions into longitude/latitude with `pyproj`.

One important wrinkle: longitude is not a single 1D coordinate for this sinusoidal tile.
Each row has a slightly different lon spacing, so we will attach **2D lon/lat coordinates**
to the xarray DataArray before plotting.

In [ ]:
from pyproj import CRS, Transformer

# Projection/grid parameters from the HDF metadata
R = 6371007.181          # MODIS sinusoidal sphere radius (m)
x_ul = -7783653.637667   # upper-left X corner (m)
y_ul =  5559752.598333   # upper-left Y corner (m)
ps = 463.312716527778    # pixel size (m)

# Pixel-center coordinates in the native MODIS sinusoidal grid
n_y = snow_clean.sizes['y']
n_x = snow_clean.sizes['x']
x_centers = x_ul + (np.arange(n_x) + 0.5) * ps
y_centers = y_ul - (np.arange(n_y) + 0.5) * ps
x2d, y2d = np.meshgrid(x_centers, y_centers)

# Convert MODIS sinusoidal meters to lon/lat degrees
modis_crs = CRS.from_proj4(f'+proj=sinu +R={R} +nadgrids=@null +wktext')
lonlat_crs = CRS.from_epsg(4326)
transformer = Transformer.from_crs(modis_crs, lonlat_crs, always_xy=True)
lon2d, lat2d = transformer.transform(x2d, y2d)

# Attach the 2D coordinates to the same xarray DataArray
snow_geo = snow_clean.assign_coords(
    lon=(('y', 'x'), lon2d),
    lat=(('y', 'x'), lat2d)
)

# Also attach the native projected x/y coordinates. These are useful for transformations,
# but the final figure below will use pixel coordinates to preserve the original tile shape.
snow_map = snow_clean.assign_coords(
    x=('x', x_centers),
    y=('y', y_centers)
)

print(f'Lat range: {np.nanmin(lat2d):.2f}° to {np.nanmax(lat2d):.2f}°N')
print(f'Lon range: {np.nanmin(lon2d):.2f}° to {np.nanmax(lon2d):.2f}°E')
print('snow_geo dims:', snow_geo.dims)
print('snow_geo coords:', list(snow_geo.coords))
print('snow_map coords:', list(snow_map.coords))

In [ ]:
# Make a MATLAB-style parula colormap
from matplotlib.colors import LinearSegmentedColormap

# Sample subset of official MATLAB Parula RGB values
parula_data = [
    (0.2081, 0.1663, 0.5292),
    (0.0591, 0.3598, 0.8683),
    (0.0231, 0.6418, 0.7913),
    (0.1801, 0.7177, 0.6424),
    (0.5300, 0.7491, 0.4661),
    (0.9139, 0.7258, 0.3063),
    (0.9763, 0.9831, 0.0538)
]

parula_cmap = LinearSegmentedColormap.from_list('Parula', parula_data)

In [ ]:
import geopandas as gpd
import cartopy.io.shapereader as shpreader

# Load Great Lakes from Natural Earth (cached locally by cartopy after first download)
lakes_shp = shpreader.natural_earth(resolution='10m', category='physical', name='lakes')
lakes = gpd.read_file(lakes_shp)
great_lakes = lakes[lakes['name'].isin([
    'Lake Superior', 'Lake Michigan', 'Lake Huron', 'Lake Erie', 'Lake Ontario'
])]

# Plot: xarray handles the snow data, geopandas handles the lake outlines
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)

mesh = snow_geo.plot.pcolormesh(
    ax=ax,
    x='lon',
    y='lat',
    cmap=parula_cmap,
    vmin=0,
    vmax=85,
    add_colorbar=False,
    rasterized=True
)
great_lakes.plot(ax=ax, facecolor='lightcyan', edgecolor='steelblue', linewidth=1.5,
                 zorder=3, label='Great Lakes')

ax.set_title('MODIS Cloud-Gap-Filled Snow Cover with Great Lakes\nTile h11v04 - Feb 15, 2026')
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
ax.set_xlim(np.nanmin(lon2d), np.nanmax(lon2d))
ax.set_ylim(np.nanmin(lat2d), np.nanmax(lat2d))
ax.set_aspect('equal')

cbar = fig.colorbar(mesh, ax=ax, orientation='horizontal', pad=0.08,
                    fraction=0.055, aspect=40)
cbar.set_label('Snow cover (%)')

# Optional: try Natural Earth state/province lines too
# states_shp = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_1_states_provinces_lines')
# gpd.read_file(states_shp).plot(ax=ax, color='0.25', linewidth=0.4, zorder=4)
plt.show()

## Wrapping up

The three most important ideas from today:

1. **Dataset = library, DataArray = checked-out book.** Open files as Datasets, then pull out the variable you want to work with.
2. **DataArrays keep labels attached.** `dims` name the axes, and `coords` hold the coordinate values along those axes.
3. **Use labels and dimension names.** `.sel(lat=29.0, method='nearest')` and `.mean(dim='lon')` are easier to read and less error-prone than raw integer indexing.


### Exit ticket

Use the OISST file and the same DataArray-first workflow we used all lesson.


In [ ]:
# Exit ticket
# 1. Open the OISST file as a Dataset.
# 2. Check out the SST DataArray.
# 3. Squeeze it to 2D.
# 4. Select the North Atlantic: lat 30-65°N, lon 280-360°E.
# 5. Compute the regional mean SST.
# 6. Plot only grid cells warmer than the regional mean.
# 7. In one sentence, explain the difference between .isel(lat=0) and
#    .sel(lat=0.0, method='nearest').

ds_exit = xr.open_dataset('data/oisst_sst_20220304.nc')
sst_exit = ds_exit['sst'].squeeze()

natl = sst_exit.sel(lat=slice(30, 65), lon=slice(280, 360))
natl_mean = float(natl.mean(dim=['lat', 'lon']))
print(f'North Atlantic mean SST: {natl_mean:.2f} °C')

natl.where(natl > natl_mean).plot(cmap='RdYlBu_r', figsize=(10, 5))
plt.title(f'North Atlantic SST above regional mean ({natl_mean:.1f} °C)')
plt.show()

# .isel(lat=0) selects by integer position, so it means the first latitude row.
# .sel(lat=0.0, method='nearest') selects by coordinate value, so it means the
# latitude row closest to 0.0°N.
